In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, recall_score, f1_score

# 1. Load Data
df = pd.read_csv('demand_agent_training.csv')
df['week_start_date'] = pd.to_datetime(df['week_start_date'])

# --- Preprocessing Steps ---
df['festival_event'] = df['festival_event'].fillna('None')
df = df.sort_values(['week_start_date', 'location_id', 'product_id']).reset_index(drop=True)

# 2. Feature Selection & Categorical Encoding
target_col = 'next_week_demand_target_units'
feature_cols = [
    'year', 'week_number', 'demand_rank', 'weekday_units_sold', 'weekend_units_sold', 
    'units_sold', 'avg_selling_price_rs', 'promotion_flag', 'discount_pct', 'holiday_flag', 
    'temperature_c', 'rainfall_mm', 'humidity_pct', 'stockout_flag',
    'location_id', 'product_id', 'category_name', 'brand', 'season', 'weather_condition',
    'festival_event'
]

cat_cols = ['location_id', 'product_id', 'category_name', 'brand', 'season', 'weather_condition', 'festival_event']

# Include 'region_of_kolkata' and 'product_name' for reporting
data = df[feature_cols + [target_col, 'week_start_date', 'region_of_kolkata', 'product_name']].copy()
for col in cat_cols:
    data[col] = data[col].astype('category')

# 3. Chronological Time-Series Split (80% Train, 20% Test)
unique_dates = sorted(data['week_start_date'].unique())
train_cutoff = unique_dates[int(len(unique_dates) * 0.8)]

train_data = data[data['week_start_date'] <= train_cutoff]
test_data = data[data['week_start_date'] > train_cutoff]

X_train, y_train = train_data[feature_cols], train_data[target_col]
X_test, y_test = test_data[feature_cols], test_data[target_col]

# 4. Fit Model (Regressor)
model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    random_state=42
)
model.fit(X_train, y_train)

# 5. Predict & Evaluate Regression Metrics (RMSE, MAE, R²)
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Regression Metrics ---")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test MAE:  {mae:.2f}")
print(f"Test R²:   {r2:.4f}")

# 6. Evaluate Classification Metrics via Thresholding (Accuracy, Recall, F1-Score)
threshold = y_train.median()
y_test_binary = (y_test > threshold).astype(int)
y_pred_binary = (y_pred > threshold).astype(int)

accuracy = accuracy_score(y_test_binary, y_pred_binary)
recall = recall_score(y_test_binary, y_pred_binary)
f1 = f1_score(y_test_binary, y_pred_binary)

print("\n--- Classification Metrics (Thresholded at Median Demand) ---")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Recall:   {recall:.4f}")
print(f"Test F1-Score: {f1:.4f}")

# 7. Generate and Print Location-Wise Predictions & Top Demanded Product for Next Week
test_data = test_data.copy()
test_data['predicted_demand'] = y_pred

latest_test_week = test_data['week_start_date'].max()
next_week_subset = test_data[test_data['week_start_date'] == latest_test_week]

# Find top (most demanded) product per location using idxmax()
top_idx = next_week_subset.groupby(['location_id', 'region_of_kolkata'])['predicted_demand'].idxmax()
top_products = next_week_subset.loc[top_idx, ['location_id', 'product_name', 'predicted_demand']].copy()
top_products.rename(columns={'product_name': 'top_product_name', 'predicted_demand': 'top_product_demand_units'}, inplace=True)

# Aggregate overall totals per location
location_forecast = next_week_subset.groupby(['location_id', 'region_of_kolkata'])['predicted_demand'].agg(['sum', 'mean', 'count']).reset_index()
location_forecast.rename(columns={'sum': 'total_predicted_demand_units', 'mean': 'avg_product_demand_units', 'count': 'active_products_count'}, inplace=True)
location_forecast['total_predicted_demand_units'] = location_forecast['total_predicted_demand_units'].round(2)
location_forecast['avg_product_demand_units'] = location_forecast['avg_product_demand_units'].round(2)

# Merge the top product info back into the location summary table
final_location_forecast = pd.merge(location_forecast, top_products[['location_id', 'top_product_name', 'top_product_demand_units']], on='location_id')
final_location_forecast['top_product_demand_units'] = final_location_forecast['top_product_demand_units'].round(2)

print(f"\n--- Location-Wise & Top Product Demand Predictions for Next Week (Week Starting: {latest_test_week.strftime('%Y-%m-%d')}) ---")
print(final_location_forecast.to_string(index=False))

--- Regression Metrics ---
Test RMSE: 20.60
Test MAE:  14.94
Test R²:   0.7810

--- Classification Metrics (Thresholded at Median Demand) ---
Test Accuracy: 0.8472
Test Recall:   0.8100
Test F1-Score: 0.8461

--- Location-Wise & Top Product Demand Predictions for Next Week (Week Starting: 2026-08-23) ---
location_id region_of_kolkata  total_predicted_demand_units  avg_product_demand_units  active_products_count          top_product_name  top_product_demand_units
KOL-LOC-001      Lake Gardens                    411.179993                     82.24                      5    Maggi 2-Minute Noodles                100.870003
KOL-LOC-002          Jadavpur                    567.919983                    113.58                      5    Maggi 2-Minute Noodles                131.000000
KOL-LOC-003          Dhakuria                    411.959991                     82.39                      5    Maggi 2-Minute Noodles                117.870003
KOL-LOC-004          Gariahat                    5

C:\Users\Koushik Hazra\AppData\Local\Temp\ipykernel_17192\1927769087.py:89: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  top_products.rename(columns={'product_name': 'top_product_name', 'predicted_demand': 'top_product_demand_units'}, inplace=True)
C:\Users\Koushik Hazra\AppData\Local\Temp\ipykernel_17192\1927769087.py:93: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  location_forecast.rename(columns={'sum': 'total_predicted_demand_units', 'mean': 'avg_product_demand_units', 'count': 'active_products_count'}, inplace=True)
